# La Liga EDA — StatsBomb Open Data

In-depth exploratory data analysis of La Liga matches available in the StatsBomb open dataset.
Covers the full range of available seasons (Messi-era Barcelona, roughly 2004/05 – 2019/20).

**Focus areas:**
- groupby aggregations
- value_counts
- Plotly visualisations

**Sections:**
1. Setup & Data Loading
2. Seasons & Matches Overview
3. Team-Level Analysis
4. Player-Level Analysis
5. Shot & xG Analysis
6. Pass Analysis
7. Match Tempo & Event Volume
8. Home vs Away

---
## 1. Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsbombpy import sb

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

# La Liga competition ID in StatsBomb
LA_LIGA_ID = 11

In [ ]:
# Get all available La Liga seasons
competitions = sb.competitions()
la_liga_seasons = competitions[competitions['competition_id'] == LA_LIGA_ID].copy()
la_liga_seasons = la_liga_seasons.sort_values('season_name')[['season_id', 'season_name', 'competition_gender']]

print(f"Available La Liga seasons: {len(la_liga_seasons)}")
la_liga_seasons

In [ ]:
# Load matches for all seasons
all_matches = []
for _, row in la_liga_seasons.iterrows():
    season_matches = sb.matches(competition_id=LA_LIGA_ID, season_id=row['season_id'])
    season_matches['season_name'] = row['season_name']
    all_matches.append(season_matches)

matches = pd.concat(all_matches, ignore_index=True)
print(f"Total matches loaded: {len(matches)}")
matches.head()

In [ ]:
# Load all events across every match — this will take a few minutes
print("Loading events for all matches...")
all_events = []
for _, match in matches.iterrows():
    match_events = sb.events(match_id=match['match_id'])
    match_events['match_id']    = match['match_id']
    match_events['season_name'] = match['season_name']
    match_events['home_team']   = match['home_team']
    match_events['away_team']   = match['away_team']
    all_events.append(match_events)

events = pd.concat(all_events, ignore_index=True)
print(f"Total events loaded: {len(events):,}")
print(f"Columns: {len(events.columns)}")

---
## 2. Seasons & Matches Overview

In [ ]:
# Matches per season
matches_per_season = (
    matches.groupby('season_name')
    .agg(total_matches=('match_id', 'count'))
    .reset_index()
    .sort_values('season_name')
)
print(matches_per_season.to_string(index=False))

In [ ]:
fig = px.bar(
    matches_per_season,
    x='season_name',
    y='total_matches',
    title='Matches Available per Season',
    labels={'season_name': 'Season', 'total_matches': 'Matches'},
    text='total_matches'
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Goals per match over seasons
goals_per_season = (
    matches.groupby('season_name')
    .agg(
        total_home_goals=('home_score', 'sum'),
        total_away_goals=('away_score', 'sum'),
        total_matches=('match_id', 'count')
    )
    .reset_index()
)
goals_per_season['total_goals'] = goals_per_season['total_home_goals'] + goals_per_season['total_away_goals']
goals_per_season['goals_per_match'] = (goals_per_season['total_goals'] / goals_per_season['total_matches']).round(2)
goals_per_season = goals_per_season.sort_values('season_name')
goals_per_season

In [ ]:
fig = px.line(
    goals_per_season,
    x='season_name',
    y='goals_per_match',
    title='Average Goals per Match by Season',
    labels={'season_name': 'Season', 'goals_per_match': 'Goals per Match'},
    markers=True
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Match outcome distribution
matches['result'] = matches.apply(
    lambda r: 'Home Win' if r['home_score'] > r['away_score']
              else ('Away Win' if r['home_score'] < r['away_score'] else 'Draw'),
    axis=1
)
result_counts = matches['result'].value_counts().reset_index()
result_counts.columns = ['result', 'count']
print(result_counts)

In [ ]:
fig = px.pie(
    result_counts,
    names='result',
    values='count',
    title='Match Outcome Distribution (All Seasons)',
    color='result',
    color_discrete_map={'Home Win': '#1f77b4', 'Away Win': '#ff7f0e', 'Draw': '#2ca02c'}
)
fig.show()

---
## 3. Team-Level Analysis

In [ ]:
# Build a team results table from home and away perspectives
home = matches[['home_team', 'away_team', 'home_score', 'away_score', 'result']].copy()
home.columns = ['team', 'opponent', 'gf', 'ga', 'raw_result']
home['win']  = (home['raw_result'] == 'Home Win').astype(int)
home['draw'] = (home['raw_result'] == 'Draw').astype(int)
home['loss'] = (home['raw_result'] == 'Away Win').astype(int)

away = matches[['away_team', 'home_team', 'away_score', 'home_score', 'result']].copy()
away.columns = ['team', 'opponent', 'gf', 'ga', 'raw_result']
away['win']  = (away['raw_result'] == 'Away Win').astype(int)
away['draw'] = (away['raw_result'] == 'Draw').astype(int)
away['loss'] = (away['raw_result'] == 'Home Win').astype(int)

team_results = pd.concat([home, away], ignore_index=True)

team_summary = (
    team_results.groupby('team')
    .agg(
        matches=('win', 'count'),
        wins=('win', 'sum'),
        draws=('draw', 'sum'),
        losses=('loss', 'sum'),
        goals_for=('gf', 'sum'),
        goals_against=('ga', 'sum')
    )
    .reset_index()
)
team_summary['points']   = team_summary['wins'] * 3 + team_summary['draws']
team_summary['goal_diff'] = team_summary['goals_for'] - team_summary['goals_against']
team_summary['win_pct']  = (team_summary['wins'] / team_summary['matches'] * 100).round(1)
team_summary = team_summary.sort_values('points', ascending=False)

print("Top 15 teams by total points across all seasons:")
team_summary.head(15)

In [ ]:
fig = px.bar(
    team_summary.head(15),
    x='team',
    y='points',
    title='Top 15 Teams by Total Points (All Seasons Combined)',
    labels={'team': 'Team', 'points': 'Total Points'},
    color='win_pct',
    color_continuous_scale='Blues',
    text='points'
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Goals scored vs conceded (top 15 teams)
top_teams = team_summary.head(15)

fig = go.Figure()
fig.add_trace(go.Bar(name='Goals For',     x=top_teams['team'], y=top_teams['goals_for'],     marker_color='steelblue'))
fig.add_trace(go.Bar(name='Goals Against', x=top_teams['team'], y=top_teams['goals_against'], marker_color='tomato'))
fig.update_layout(
    barmode='group',
    title='Goals Scored vs Conceded — Top 15 Teams',
    xaxis_tickangle=-45,
    xaxis_title='Team',
    yaxis_title='Goals'
)
fig.show()

In [ ]:
# Win % scatter vs goals scored
fig = px.scatter(
    team_summary[team_summary['matches'] >= 20],
    x='goals_for',
    y='win_pct',
    text='team',
    size='matches',
    color='goal_diff',
    color_continuous_scale='RdYlGn',
    title='Goals Scored vs Win % (teams with 20+ matches)',
    labels={'goals_for': 'Total Goals Scored', 'win_pct': 'Win %', 'goal_diff': 'Goal Diff'}
)
fig.update_traces(textposition='top center')
fig.show()

---
## 4. Player-Level Analysis

In [ ]:
# Events per player — total touches
player_events = (
    events.groupby(['player', 'team'])
    .agg(total_events=('id', 'count'))
    .reset_index()
    .sort_values('total_events', ascending=False)
)
print("Top 20 most active players (by event count):")
player_events.head(20)

In [ ]:
# What event types does each top player produce?
top_players = player_events.head(10)['player'].tolist()

player_event_types = (
    events[events['player'].isin(top_players)]
    .groupby(['player', 'type'])
    .agg(count=('id', 'count'))
    .reset_index()
)

fig = px.bar(
    player_event_types,
    x='player',
    y='count',
    color='type',
    title='Event Type Breakdown — Top 10 Players',
    labels={'count': 'Events', 'player': 'Player'},
    barmode='stack'
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

In [ ]:
# Goals by player (shots that resulted in a goal)
shots = events[events['type'] == 'Shot'].copy()

goals_by_player = (
    shots[shots['shot_outcome'] == 'Goal']
    .groupby(['player', 'team'])
    .agg(goals=('id', 'count'))
    .reset_index()
    .sort_values('goals', ascending=False)
)
print("Top 20 scorers across all seasons:")
goals_by_player.head(20)

In [ ]:
fig = px.bar(
    goals_by_player.head(20),
    x='goals',
    y='player',
    orientation='h',
    color='team',
    title='Top 20 Goal Scorers — All La Liga Seasons',
    labels={'goals': 'Goals', 'player': 'Player'}
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# Goals by season for the top 5 scorers
top5_scorers = goals_by_player.head(5)['player'].tolist()

goals_by_season = (
    shots[
        (shots['shot_outcome'] == 'Goal') &
        (shots['player'].isin(top5_scorers))
    ]
    .groupby(['player', 'season_name'])
    .agg(goals=('id', 'count'))
    .reset_index()
    .sort_values('season_name')
)

fig = px.line(
    goals_by_season,
    x='season_name',
    y='goals',
    color='player',
    title='Goals per Season — Top 5 Scorers',
    labels={'season_name': 'Season', 'goals': 'Goals'},
    markers=True
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

---
## 5. Shot & xG Analysis

In [ ]:
# Shot outcome breakdown
shot_outcomes = shots['shot_outcome'].value_counts().reset_index()
shot_outcomes.columns = ['outcome', 'count']
print(shot_outcomes)

fig = px.pie(
    shot_outcomes,
    names='outcome',
    values='count',
    title='Shot Outcome Distribution'
)
fig.show()

In [ ]:
# Shot technique breakdown
print("Shot techniques:")
print(shots['shot_technique'].value_counts())

In [ ]:
# Shot body part breakdown
print("Shot body parts:")
print(shots['shot_body_part'].value_counts())

In [ ]:
# Conversion rate by shot type
shot_type_conv = (
    shots.groupby('shot_type')
    .agg(
        total_shots=('id', 'count'),
        goals=('shot_outcome', lambda x: (x == 'Goal').sum())
    )
    .reset_index()
)
shot_type_conv['conversion_pct'] = (shot_type_conv['goals'] / shot_type_conv['total_shots'] * 100).round(1)
shot_type_conv = shot_type_conv.sort_values('conversion_pct', ascending=False)
print(shot_type_conv)

In [ ]:
# xG distribution: goals vs non-goals
shots_with_xg = shots[shots['shot_statsbomb_xg'].notna()].copy()
shots_with_xg['outcome_label'] = shots_with_xg['shot_outcome'].apply(
    lambda x: 'Goal' if x == 'Goal' else 'No Goal'
)

fig = px.histogram(
    shots_with_xg,
    x='shot_statsbomb_xg',
    color='outcome_label',
    nbins=40,
    barmode='overlay',
    opacity=0.7,
    title='xG Distribution — Goals vs No Goals',
    labels={'shot_statsbomb_xg': 'xG', 'outcome_label': 'Outcome'},
    color_discrete_map={'Goal': 'green', 'No Goal': 'tomato'}
)
fig.show()

In [ ]:
# xG vs actual goals per team (over/underperformance)
team_xg = (
    shots_with_xg.groupby('team')
    .agg(
        total_xg=('shot_statsbomb_xg', 'sum'),
        actual_goals=('shot_outcome', lambda x: (x == 'Goal').sum()),
        total_shots=('id', 'count')
    )
    .reset_index()
)
team_xg['xg_diff'] = (team_xg['actual_goals'] - team_xg['total_xg']).round(1)
team_xg['total_xg'] = team_xg['total_xg'].round(1)
team_xg = team_xg.sort_values('actual_goals', ascending=False)

print("Top 15 teams — xG vs actual goals:")
team_xg.head(15)

In [ ]:
fig = px.scatter(
    team_xg[team_xg['total_shots'] >= 50],
    x='total_xg',
    y='actual_goals',
    text='team',
    color='xg_diff',
    color_continuous_scale='RdYlGn',
    title='xG vs Actual Goals by Team (50+ shots)',
    labels={'total_xg': 'Total xG', 'actual_goals': 'Actual Goals', 'xg_diff': 'Goals - xG'}
)
# Add diagonal reference line (perfect calibration)
max_val = max(team_xg['total_xg'].max(), team_xg['actual_goals'].max())
fig.add_shape(type='line', x0=0, y0=0, x1=max_val, y1=max_val,
              line=dict(color='gray', dash='dash'))
fig.update_traces(textposition='top center')
fig.show()

In [ ]:
# xG per player — who generates the most quality chances?
player_xg = (
    shots_with_xg.groupby(['player', 'team'])
    .agg(
        total_xg=('shot_statsbomb_xg', 'sum'),
        total_shots=('id', 'count'),
        goals=('shot_outcome', lambda x: (x == 'Goal').sum())
    )
    .reset_index()
)
player_xg['xg_per_shot'] = (player_xg['total_xg'] / player_xg['total_shots']).round(3)
player_xg['goals_minus_xg'] = (player_xg['goals'] - player_xg['total_xg']).round(1)
player_xg = player_xg.sort_values('total_xg', ascending=False)

print("Top 15 players by total xG:")
player_xg.head(15)

In [ ]:
fig = px.bar(
    player_xg.head(15),
    x='player',
    y=['total_xg', 'goals'],
    barmode='group',
    title='Total xG vs Actual Goals — Top 15 Players',
    labels={'value': 'Count', 'player': 'Player', 'variable': 'Metric'}
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

---
## 6. Pass Analysis

In [ ]:
passes = events[events['type'] == 'Pass'].copy()
print(f"Total passes: {len(passes):,}")

# Pass outcome breakdown
print("\nPass outcomes:")
print(passes['pass_outcome'].value_counts())
print("\n(NaN = successful pass — StatsBomb only records outcome when incomplete)")

In [ ]:
# Pass completion rate by team
passes['completed'] = passes['pass_outcome'].isna().astype(int)

team_passing = (
    passes.groupby('team')
    .agg(
        total_passes=('id', 'count'),
        completed_passes=('completed', 'sum')
    )
    .reset_index()
)
team_passing['completion_pct'] = (team_passing['completed_passes'] / team_passing['total_passes'] * 100).round(1)
team_passing = team_passing.sort_values('completion_pct', ascending=False)

print("Pass completion by team (top 15):")
team_passing.head(15)

In [ ]:
fig = px.bar(
    team_passing.head(15),
    x='team',
    y='completion_pct',
    title='Pass Completion % — Top 15 Teams',
    labels={'team': 'Team', 'completion_pct': 'Completion %'},
    color='completion_pct',
    color_continuous_scale='Blues',
    text='completion_pct'
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Pass height breakdown
print("Pass height:")
print(passes['pass_height'].value_counts())

In [ ]:
# Pass length distribution
fig = px.histogram(
    passes[passes['pass_length'].notna()],
    x='pass_length',
    nbins=50,
    title='Pass Length Distribution',
    labels={'pass_length': 'Pass Length (metres)'}
)
fig.show()

In [ ]:
# Average pass length by team
team_pass_length = (
    passes[passes['pass_length'].notna()]
    .groupby('team')
    .agg(avg_pass_length=('pass_length', 'mean'))
    .reset_index()
    .sort_values('avg_pass_length', ascending=False)
)
team_pass_length['avg_pass_length'] = team_pass_length['avg_pass_length'].round(1)

fig = px.bar(
    team_pass_length.head(15),
    x='team',
    y='avg_pass_length',
    title='Average Pass Length — Top 15 Teams',
    labels={'team': 'Team', 'avg_pass_length': 'Avg Pass Length (m)'},
    text='avg_pass_length'
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Top passers by volume
player_passing = (
    passes.groupby(['player', 'team'])
    .agg(
        total_passes=('id', 'count'),
        completed=('completed', 'sum'),
        avg_length=('pass_length', 'mean')
    )
    .reset_index()
)
player_passing['completion_pct'] = (player_passing['completed'] / player_passing['total_passes'] * 100).round(1)
player_passing['avg_length'] = player_passing['avg_length'].round(1)
player_passing = player_passing.sort_values('total_passes', ascending=False)

print("Top 15 passers by volume:")
player_passing.head(15)

---
## 7. Match Tempo & Event Volume

In [ ]:
# Overall event type distribution
event_type_counts = events['type'].value_counts().reset_index()
event_type_counts.columns = ['event_type', 'count']
print(event_type_counts)

In [ ]:
fig = px.bar(
    event_type_counts,
    x='count',
    y='event_type',
    orientation='h',
    title='Event Type Distribution — All La Liga Matches',
    labels={'count': 'Count', 'event_type': 'Event Type'}
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# Events by minute — when does play happen?
events_by_minute = (
    events[events['minute'].notna()]
    .groupby('minute')
    .agg(count=('id', 'count'))
    .reset_index()
)

fig = px.line(
    events_by_minute[events_by_minute['minute'] <= 95],
    x='minute',
    y='count',
    title='Total Events by Minute (all matches combined)',
    labels={'minute': 'Minute', 'count': 'Total Events'}
)
fig.show()

In [ ]:
# Goals by minute
goals = shots[shots['shot_outcome'] == 'Goal'].copy()

goals_by_minute = (
    goals[goals['minute'] <= 95]
    .groupby('minute')
    .agg(goals=('id', 'count'))
    .reset_index()
)

fig = px.bar(
    goals_by_minute,
    x='minute',
    y='goals',
    title='Goals by Match Minute',
    labels={'minute': 'Minute', 'goals': 'Goals'}
)
fig.show()

In [ ]:
# Events per match by season (tempo over time)
events_per_match = (
    events.groupby(['match_id', 'season_name'])
    .agg(event_count=('id', 'count'))
    .reset_index()
)

fig = px.box(
    events_per_match,
    x='season_name',
    y='event_count',
    title='Events per Match Distribution by Season',
    labels={'season_name': 'Season', 'event_count': 'Events per Match'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

---
## 8. Home vs Away

In [ ]:
# Home vs away goals per season
home_away = (
    matches.groupby('season_name')
    .agg(
        home_goals=('home_score', 'sum'),
        away_goals=('away_score', 'sum'),
        matches=('match_id', 'count')
    )
    .reset_index()
)
home_away['home_gpg'] = (home_away['home_goals'] / home_away['matches']).round(2)
home_away['away_gpg'] = (home_away['away_goals'] / home_away['matches']).round(2)
home_away = home_away.sort_values('season_name')

fig = go.Figure()
fig.add_trace(go.Scatter(x=home_away['season_name'], y=home_away['home_gpg'], name='Home GPM', mode='lines+markers'))
fig.add_trace(go.Scatter(x=home_away['season_name'], y=home_away['away_gpg'], name='Away GPM', mode='lines+markers'))
fig.update_layout(
    title='Home vs Away Goals per Match by Season',
    xaxis_title='Season',
    yaxis_title='Goals per Match',
    xaxis_tickangle=-45
)
fig.show()

In [ ]:
# Home advantage: result distribution by season
result_by_season = (
    matches.groupby(['season_name', 'result'])
    .agg(count=('match_id', 'count'))
    .reset_index()
    .sort_values('season_name')
)

fig = px.bar(
    result_by_season,
    x='season_name',
    y='count',
    color='result',
    title='Match Results by Season',
    labels={'season_name': 'Season', 'count': 'Matches', 'result': 'Result'},
    barmode='stack',
    color_discrete_map={'Home Win': '#1f77b4', 'Away Win': '#ff7f0e', 'Draw': '#2ca02c'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()